# Monte Carlo Option Pricing Simulation

This notebook is a **thin orchestrator** that delegates logic to modular packages:

| Module | Responsibility |
| --- | --- |
| `config/settings.py` | Table names, export paths, logging |
| `transforms/simulation.py` | Random generators, simulation functions, UDF factories |
| `utils/simulation_helpers.py` | Data loading, distribution fitting, broadcasting, aggregation, writing |

**Parameters (widgets):** ticker, strike price (K), simulation period (T days), number of runs

**Input:** `default.yfinance_historical_data`  
**Output:** `default.simulation_results`

In [0]:
%pip install scipy --quiet
dbutils.library.restartPython()

In [0]:
import sys
import os

# Ensure the src directory is on the Python path
src_dir = "/Workspace/Users/spyros.louvis@ctpsandbox.com/pub-python-repo/databricks/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Import pipeline modules
from config.settings import FULL_TABLE_NAME, FULL_RESULTS_TABLE_NAME, get_export_path, get_logger
from transforms.simulation import (
    init_broadcast,
    udf_historical, udf_expon, udf_cauchy, udf_t,
    udf_window, udf_ten_window, udf_twenty_window,
)
from utils.simulation_helpers import (
    load_historical_data, fit_distributions, build_and_broadcast,
    create_target_dataframe, run_all_simulations, aggregate_results, write_results,
)

logger = get_logger("monte_carlo_simulation")
logger.info("Modules loaded successfully.")

In [0]:
# ---------------------------------------------------------------------------
# Widgets (parameterized for Job runs or interactive use)
# ---------------------------------------------------------------------------
dbutils.widgets.text("ticker", "^GSPC", "Ticker Symbol")
dbutils.widgets.text("target_value", "5700", "Strike Price K")
dbutils.widgets.text("period", "10", "Simulation Period T (days)")
dbutils.widgets.text("num_simulations", "1000", "Number of Simulations")

ticker = dbutils.widgets.get("ticker")
K = float(dbutils.widgets.get("target_value"))
T = int(dbutils.widgets.get("period"))
num_runs = int(dbutils.widgets.get("num_simulations"))

logger.info(f"Parameters: ticker={ticker}, K={K}, T={T}, runs={num_runs}")

In [0]:
# ---------------------------------------------------------------------------
# Load data, fit distributions, and broadcast
# ---------------------------------------------------------------------------
historical_df, S0 = load_historical_data(spark, FULL_TABLE_NAME, ticker)
dist_params = fit_distributions(historical_df)
broadcasts = build_and_broadcast(spark, historical_df, dist_params)

# Initialize the simulation module with broadcast variables
init_broadcast(
    broadcasts["b"], broadcasts["b_loc"], broadcasts["b_scale"],
    broadcasts["b_deg_f"], broadcasts["b_tloc"], broadcasts["b_tscale"],
    broadcasts["b_mu"], broadcasts["b_std"],
)
logger.info("Simulation module initialized.")

In [0]:
# ---------------------------------------------------------------------------
# Run all simulations
# ---------------------------------------------------------------------------
target_df = create_target_dataframe(spark, K, T, num_runs)

# Define simulation methods to run
udf_map = {
    "historical": udf_historical,
    "window": udf_window,
    "window_10d": udf_ten_window,
    "window_20d": udf_twenty_window,
    "exponential": udf_expon,
    "cauchy": udf_cauchy,
    "student_t": udf_t,
}

simulation_results = run_all_simulations(target_df, S0, udf_map)
logger.info("All simulations defined.")

In [0]:
# ---------------------------------------------------------------------------
# Aggregate and display results
# ---------------------------------------------------------------------------
agg_results = aggregate_results(simulation_results, ticker, S0)
display(agg_results)

In [0]:
# ---------------------------------------------------------------------------
# Write results to Delta table + Parquet export
# ---------------------------------------------------------------------------
export_path = get_export_path(ticker)

write_results(agg_results, FULL_RESULTS_TABLE_NAME, ticker, export_path)

logger.info(f"[DONE] Simulation complete for ticker={ticker}, K={K}, T={T}, runs={num_runs}")